# 06 — Framework overview

An executable tour of the `disp_s1_eval` framework. The notebook is short and self-contained: no data downloads, no Earthdata or NGL credentials. It exercises the core abstractions — processor registry, GNSS ingestion, ENU-to-LOS projection, error diagnostics, experiment dry-run — in a clean environment.

## 1. Imports and version

In [ ]:
import disp_s1_eval
from disp_s1_eval.processors import available_readers

print('disp_s1_eval version:', disp_s1_eval.__version__)
print('Registered readers:', available_readers())

## 2. GNSS ingestion and ENU-to-LOS projection

We parse a small inline `tenv3` sample, build the per-epoch ENU covariance, and project both the displacement and its covariance into a representative Sentinel-1 ascending LOS direction (incidence ≈ 38°, heading ≈ −12°).

In [ ]:
import numpy as np
from disp_s1_eval.gnss import parse_tenv3, project_enu_to_los, project_enu_covariance_to_los

_tenv3 = (
    'P056 18JAN01 2018.0014 58119 1980 1 -120.0 -2360000 0.001 4170000 0.002 100 0.003 0.000 0.0010 0.0010 0.0030 0.10 0.05 -0.20\n'
    'P056 18JAN02 2018.0041 58120 1980 2 -120.0 -2360000 0.002 4170000 0.001 100 0.005 0.000 0.0010 0.0010 0.0030 0.10 0.05 -0.20\n'
    'P056 18JAN03 2018.0068 58121 1980 3 -120.0 -2360000 0.003 4170000 0.000 100 0.007 0.000 0.0010 0.0010 0.0030 0.10 0.05 -0.20\n'
)
series = parse_tenv3(_tenv3.splitlines(), longitude=-120.0, latitude=36.5)

incidence_rad = np.deg2rad(38.0)
heading_rad = np.deg2rad(-12.0)
los_unit = np.array([
    -np.sin(incidence_rad) * np.cos(heading_rad - 1.5 * np.pi),
    np.sin(incidence_rad) * np.sin(heading_rad - 1.5 * np.pi),
    np.cos(incidence_rad),
])

enu = np.stack([series.east_mm, series.north_mm, series.up_mm], axis=0)
los_mm = project_enu_to_los(enu, los_unit)
var_los = np.array([
    project_enu_covariance_to_los(series.covariance(i), los_unit)
    for i in range(len(series.dates))
])
print('LOS displacement (mm):', np.round(los_mm, 3))
print('LOS 1-sigma (mm):', np.round(np.sqrt(var_los), 3))

## 3. Error diagnostics

Triple collocation on three synthetic measurements of a common signal recovers the per-product noise scales declared in the simulation.

In [ ]:
from disp_s1_eval.errors import triple_collocation

rng = np.random.default_rng(42)
truth = rng.standard_normal(2000)
x = truth + rng.standard_normal(2000) * 0.5
y = truth + rng.standard_normal(2000) * 0.8
z = truth + rng.standard_normal(2000) * 0.3
result = triple_collocation(x, y, z, name_x='InSAR-A', name_y='InSAR-B', name_z='GNSS')
print(result)

## 4. Experiment dry-run

Each experiment can be dry-run to validate its configuration and emit a `manifest.json` with provenance, software versions, and the configuration's SHA-256 hash. This is the auditable trace of every analysis the framework performs.

In [ ]:
import json
import tempfile
from pathlib import Path
from experiments.E01_central_valley_disp_s1_vs_gnss.run import main as run_e01

with tempfile.TemporaryDirectory() as tmp:
    out = Path(tmp) / 'results'
    run_e01([
        '--config', 'experiments/E01_central_valley_disp_s1_vs_gnss/config.yml',
        '--output', str(out),
        '--dry-run',
    ])
    manifest = json.loads((out / 'manifest.json').read_text(encoding='utf-8'))

for key in ('experiment', 'framework_version', 'python_version', 'platform', 'config_sha256'):
    print(f'{key}: {manifest[key]}')
print('config keys:', sorted(manifest['config']))

## Scope

This is an overview, not a result. It does not download OPERA DISP-S1 granules, fetch NGL `tenv3` files, or compute validation metrics on real data.

Real-data analyses live under `experiments/`. Each experiment is reproducible from a single CLI command and writes its full provenance to `results/manifest.json`.